In [1]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans

import re
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/adnanaltimeemy/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [2]:
df = pd.read_csv("Yelp-businesses-reviews.csv")

# Show structure
print(df.head())
print(df.columns)

                            business_id          Date  Rating  \
0                demergan-wilton-manors  "09/08/2024"       5   
1  supreme-pool-tile-cleaning-la-quinta  "09/04/2024"       5   
2  supreme-pool-tile-cleaning-la-quinta  "05/08/2022"       5   
3  leisure-pools-of-sw-florida-naples-2  "27/02/2023"       1   
4  leisure-pools-of-sw-florida-naples-2  "05/01/2023"       1   

                                             Content  \
0  Our insurance mandated we replace our 15 year ...   
1  Rudy just cleaned and sealed our pool tiles. I...   
2  Can you say looks like a brand new pool?\n\nWe...   
3  We finally fired Leisure Pools of Naples becau...   
4  They are too busy and looks like they have eno...   

                                       Review_auther  \
0  {"Friends":434,"Image":"https://s3-media0.fl.y...   
1  {"Friends":15,"Image":"https://s3-media0.fl.ye...   
2  {"Friends":1221,"Image":"https://s3-media0.fl....   
3  {"Friends":0,"Location":"SoMa, San Francisco,

In [4]:
print(df.columns.tolist())
print(df.head())

['business_id', 'Date', 'Rating', 'Content', 'Review_auther', 'Review_image', 'Reactions', 'Replies', 'review_order', 'Eelite_status', 'check-in_status', 'business_name']
                            business_id          Date  Rating  \
0                demergan-wilton-manors  "09/08/2024"       5   
1  supreme-pool-tile-cleaning-la-quinta  "09/04/2024"       5   
2  supreme-pool-tile-cleaning-la-quinta  "05/08/2022"       5   
3  leisure-pools-of-sw-florida-naples-2  "27/02/2023"       1   
4  leisure-pools-of-sw-florida-naples-2  "05/01/2023"       1   

                                             Content  \
0  Our insurance mandated we replace our 15 year ...   
1  Rudy just cleaned and sealed our pool tiles. I...   
2  Can you say looks like a brand new pool?\n\nWe...   
3  We finally fired Leisure Pools of Naples becau...   
4  They are too busy and looks like they have eno...   

                                       Review_auther  \
0  {"Friends":434,"Image":"https://s3-media0.

In [5]:
df.head()

,business_id,Date,Rating,Content,Review_auther,Review_image,Reactions,Replies,review_order,Eelite_status,check-in_status,business_name
0,demergan-wilton-manors,"""09/08/2024""",5,Our insurance mandated we replace our 15 year ...,"{""Friends"":434,""Image"":""https://s3-media0.fl.y...",NaN,"[{""Number"":0,""Title"":""Helpful""},{""Number"":0,""T...","[{""Content"":""A heartfelt thanks Richards, you’...",1,NaN,0 check-in,Demergan
1,supreme-pool-tile-cleaning-la-quinta,"""09/04/2024""",5,Rudy just cleaned and sealed our pool tiles. I...,"{""Friends"":15,""Image"":""https://s3-media0.fl.ye...","[""https://s3-media0.fl.yelpcdn.com/bphoto/ETn2...","[{""Number"":1,""Title"":""Helpful""},{""Number"":0,""T...",NaN,3,NaN,0 check-in,Supreme Pool Tile Cleaning
2,supreme-pool-tile-cleaning-la-quinta,"""05/08/2022""",5,Can you say looks like a brand new pool?\n\nWe...,"{""Friends"":1221,""Image"":""https://s3-media0.fl....","[""https://s3-media0.fl.yelpcdn.com/bphoto/QC9c...","[{""Number"":2,""Title"":""Helpful""},{""Number"":0,""T...",NaN,10,NaN,0 check-in,Supreme Pool Tile Cleaning
3,leisure-pools-of-sw-florida-naples-2,"""27/02/2023""",1,We finally fired Leisure Pools of Naples becau...,"{""Friends"":0,""Location"":""SoMa, San Francisco, ...",NaN,"[{""Number"":0,""Title"":""Helpful""},{""Number"":0,""T...","[{""Content"":""We greatly apologize that you wer...",3,NaN,0 check-in,Leisure Pools Of SW Florida
4,leisure-pools-of-sw-florida-naples-2,"""05/01/2023""",1,They are too busy and looks like they have eno...,"{""Friends"":0,""Location"":""San Francisco, CA"",""P...",NaN,"[{""Number"":1,""Title"":""Helpful""},{""Number"":0,""T...","[{""Content"":""Hello Anna! I apologize we were u...",4,NaN,0 check-in,Leisure Pools Of SW Florida


In [7]:
print(df.head())
print(df.columns.tolist())

                            business_id          Date  Rating  \
0                demergan-wilton-manors  "09/08/2024"       5   
1  supreme-pool-tile-cleaning-la-quinta  "09/04/2024"       5   
2  supreme-pool-tile-cleaning-la-quinta  "05/08/2022"       5   
3  leisure-pools-of-sw-florida-naples-2  "27/02/2023"       1   
4  leisure-pools-of-sw-florida-naples-2  "05/01/2023"       1   

                                             Content  \
0  Our insurance mandated we replace our 15 year ...   
1  Rudy just cleaned and sealed our pool tiles. I...   
2  Can you say looks like a brand new pool?\n\nWe...   
3  We finally fired Leisure Pools of Naples becau...   
4  They are too busy and looks like they have eno...   

                                       Review_auther  \
0  {"Friends":434,"Image":"https://s3-media0.fl.y...   
1  {"Friends":15,"Image":"https://s3-media0.fl.ye...   
2  {"Friends":1221,"Image":"https://s3-media0.fl....   
3  {"Friends":0,"Location":"SoMa, San Francisco,

In [8]:
df.head()

,business_id,Date,Rating,Content,Review_auther,Review_image,Reactions,Replies,review_order,Eelite_status,check-in_status,business_name
0,demergan-wilton-manors,"""09/08/2024""",5,Our insurance mandated we replace our 15 year ...,"{""Friends"":434,""Image"":""https://s3-media0.fl.y...",NaN,"[{""Number"":0,""Title"":""Helpful""},{""Number"":0,""T...","[{""Content"":""A heartfelt thanks Richards, you’...",1,NaN,0 check-in,Demergan
1,supreme-pool-tile-cleaning-la-quinta,"""09/04/2024""",5,Rudy just cleaned and sealed our pool tiles. I...,"{""Friends"":15,""Image"":""https://s3-media0.fl.ye...","[""https://s3-media0.fl.yelpcdn.com/bphoto/ETn2...","[{""Number"":1,""Title"":""Helpful""},{""Number"":0,""T...",NaN,3,NaN,0 check-in,Supreme Pool Tile Cleaning
2,supreme-pool-tile-cleaning-la-quinta,"""05/08/2022""",5,Can you say looks like a brand new pool?\n\nWe...,"{""Friends"":1221,""Image"":""https://s3-media0.fl....","[""https://s3-media0.fl.yelpcdn.com/bphoto/QC9c...","[{""Number"":2,""Title"":""Helpful""},{""Number"":0,""T...",NaN,10,NaN,0 check-in,Supreme Pool Tile Cleaning
3,leisure-pools-of-sw-florida-naples-2,"""27/02/2023""",1,We finally fired Leisure Pools of Naples becau...,"{""Friends"":0,""Location"":""SoMa, San Francisco, ...",NaN,"[{""Number"":0,""Title"":""Helpful""},{""Number"":0,""T...","[{""Content"":""We greatly apologize that you wer...",3,NaN,0 check-in,Leisure Pools Of SW Florida
4,leisure-pools-of-sw-florida-naples-2,"""05/01/2023""",1,They are too busy and looks like they have eno...,"{""Friends"":0,""Location"":""San Francisco, CA"",""P...",NaN,"[{""Number"":1,""Title"":""Helpful""},{""Number"":0,""T...","[{""Content"":""Hello Anna! I apologize we were u...",4,NaN,0 check-in,Leisure Pools Of SW Florida


In [12]:
k = df.columns